In [8]:
from pathlib import Path
from models import CifarResNET18
import torch
import numpy as np
import random
from metrics import calibration_errors, nll_loss
from dataset_utils import get_data_loaders

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
train_loader, test_loader = get_data_loaders(dataset="cifar100")

directory_path = Path("resnet") 

seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
/Users/nihar/Desktop/Research/Similarity-Aware Label-Smoothing/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 11 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [18]:
for filepath in directory_path.iterdir():
    if filepath.is_file():  # Check if the entry is a file
        model = CifarResNET18(num_classes=100).to(device)
        model.load_state_dict(torch.load(filepath, weights_only=True))
        model.eval()

        print(filepath)
        logits_list = []
        labels_list = []

        model.eval()
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)

                logits_list.append(logits)
                labels_list.append(y)

        logits_all = torch.cat(logits_list, dim=0)
        labels_all = torch.cat(labels_list, dim=0)

        ces = calibration_errors(logits_all, labels_all, n_bins=15)
        print("ECE:", 100 * ces[0])
        print("MCE:", 100 * ces[1])
        print("NLL:", nll_loss(logits_all, labels_all))
        print()


resnet/vae4_002_1_0.pth
ECE: 2.472541853785515
MCE: 12.888522446155548
NLL: 0.8659029006958008

resnet/vae4_005_5_0.pth
ECE: 6.097767502069473
MCE: 15.31417965888977
NLL: 0.8927049040794373

resnet/vae4_002_10_0.pth
ECE: 4.463499411940575
MCE: 15.529334545135498
NLL: 0.8757277131080627

resnet/vae4_001_3_0.pth
ECE: 3.31430584192276
MCE: 14.984217286109924
NLL: 0.8546327948570251

resnet/ls001_0.pth
ECE: 4.455408826470375
MCE: 18.278001248836517
NLL: 0.893711507320404

resnet/ls_01_0.pth
ECE: 13.096646964550018
MCE: 27.454790472984314
NLL: 1.0456011295318604

resnet/ce_0.pth
ECE: 4.751721769571304
MCE: 12.923778593540192
NLL: 0.8776656985282898

resnet/ls_0005_0.pth
ECE: 4.13145087659359
MCE: 18.643248081207275
NLL: 0.894155740737915

resnet/ls005_0.pth
ECE: 8.972946554422379
MCE: 22.788920998573303
NLL: 0.9566543102264404

resnet/vae4_001_1_0.pth
ECE: 3.4765150398015976
MCE: 15.331023931503296
NLL: 0.8631239533424377

resnet/vae4_001_10_0.pth
ECE: 3.970053419470787
MCE: 17.422433197498

In [42]:
from scipy.stats import spearmanr, wilcoxon
import pandas as pd

def upper_tri_vec(D):
    idx = np.triu_indices_from(D, k=1)
    return D[idx]

def geometry_spearman(D1, D2):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)
    rho, _ = spearmanr(v1, v2)
    return rho
    
def relative_l2_change(D1, D2):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)

    return np.linalg.norm(v1 - v2) / np.linalg.norm(v1)

def permutation_test_geometry(D1, D2, n_perm=10000):
    v1 = upper_tri_vec(D1)
    v2 = upper_tri_vec(D2)
    observed = np.linalg.norm(v1 - v2)
    diffs = []

    for _ in range(n_perm):
        perm = np.random.permutation(len(v1))
        diffs.append(np.linalg.norm(v1 - v2[perm]))

    diffs = np.array(diffs)
    p_value = (np.sum(diffs >= observed) + 1) / (n_perm + 1)

    return observed, diffs.mean(), diffs.std(), p_value

def topk_similarity_score(D1, D2):
    _, topk_indices = torch.topk(-D1, 6, dim=1)
    _, topk_indices2 = torch.topk(-D2, 6, dim=1)

    commons = 0
    pos = 0
    for idx in range(100):
        arr1 = topk_indices[idx].tolist()
        arr2 = topk_indices2[idx].tolist()

        common = len(set(arr1) & set(arr2)) - 1
        count = sum(a == b for a, b in zip(arr1, arr2)) - 1
        
        commons += common
        pos += count

    return commons / 5, pos / 5


In [47]:
path = "scores/0_cifar100_latent_distances.xlsx"
D_0 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

path = "scores/05_cifar100_latent_distances.xlsx"
D_05 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

path = "scores/1_cifar100_latent_distances.xlsx"
D_1 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

path = "scores/4_cifar100_latent_distances.xlsx"
D_4 = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

pairs = [ 
    (0, 0.5, D_0, D_05),
    (0, 1, D_0, D_1),
    (0, 4, D_0, D_4),
    (0.5, 0, D_05, D_0),
    (0.5, 1, D_05, D_1),
    (0.5, 4, D_05, D_4),
    (1, 0, D_1, D_0),
    (1, 0.5, D_1, D_05),
    (1, 4, D_1, D_4),
    (4, 0, D_4, D_0),
    (4, 0.5, D_4, D_05),
    (4, 1, D_4, D_1),
          ]
for a, b, Da, Db in pairs:
    rho = geometry_spearman(Da, Db)
    delta = relative_l2_change(Da, Db)
    sim = topk_similarity_score(Da, Db)

    print(f"β {a} → {b}")
    print(f"  Spearman ρ      = {rho:.3f}")
    print(f"  Relative change = {delta:.3f}")
    print(f"  T-5 closest common = {sim[0]}; T-5 closest match = {sim[1]}")


β 0 → 0.5
  Spearman ρ      = 0.916
  Relative change = 0.468
  T-5 closest common = 68.4; T-5 closest match = 21.4
β 0 → 1
  Spearman ρ      = 0.873
  Relative change = 0.491
  T-5 closest common = 65.8; T-5 closest match = 24.0
β 0 → 4
  Spearman ρ      = 0.868
  Relative change = 0.538
  T-5 closest common = 63.8; T-5 closest match = 22.6
β 0.5 → 0
  Spearman ρ      = 0.916
  Relative change = 0.854
  T-5 closest common = 68.4; T-5 closest match = 21.4
β 0.5 → 1
  Spearman ρ      = 0.983
  Relative change = 0.065
  T-5 closest common = 82.2; T-5 closest match = 42.8
β 0.5 → 4
  Spearman ρ      = 0.977
  Relative change = 0.138
  T-5 closest common = 77.2; T-5 closest match = 38.0
β 1 → 0
  Spearman ρ      = 0.873
  Relative change = 0.924
  T-5 closest common = 65.8; T-5 closest match = 24.0
β 1 → 0.5
  Spearman ρ      = 0.983
  Relative change = 0.067
  T-5 closest common = 82.2; T-5 closest match = 42.8
β 1 → 4
  Spearman ρ      = 0.987
  Relative change = 0.105
  T-5 closest comm